## 创建自定义中间件

In [2]:
from langchain.agents.middleware import AgentMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
from langchain.agents import create_agent

model = init_chat_model(model="deepseek:deepseek-chat")


class MyMiddleware(AgentMiddleware):
    def before_model(self, state, runtime):
        """模型调用前执行"""
        print("准备调用模型")
        return None  # 返回 None 表示继续正常流程

    def after_model(self, state, runtime):
        """模型响应后执行"""
        print("模型已响应")
        return None  # 返回 None 表示不修改状态

# 使用中间件
agent = create_agent(
    model=model,
    tools=[],
    middleware=[MyMiddleware()]
)

print("\n用户: 你好")
response = agent.invoke({"messages": [{"role": "user", "content": "你好"}]})
print(f"Agent: {response['messages'][-1].content}")

D:\python_virtualenv\langchianv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



用户: 你好
准备调用模型
模型已响应
Agent: 你好！很高兴见到你！😊 我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要什么帮助，或者只是想聊聊天，我都很乐意为你提供支持！

我可以帮你解答各种问题，协助处理文档，进行创作和分析等等。有什么我可以为你做的吗？我会尽我所能热情地帮助你！✨


**日志中间件**

In [10]:
from langgraph.checkpoint.memory import InMemorySaver


class LoggingMiddleware(AgentMiddleware):
    def before_model(self, state, runtime):
        print(f"[日志] 消息数: {len(state.get('messages', []))}")
        return None

    def after_model(self, state, runtime):
        last_msg = state.get('messages', [])[-1]
        print(f"[日志] 响应类型: {last_msg.__class__.__name__}")
        return None

config = {"configurable": {"thread_id": "m1"}}

agent = create_agent(
    model=model,
    tools=[],
    middleware=[LoggingMiddleware()],
    checkpointer=InMemorySaver()
)


# print("\n用户: 你好") 
response = agent.invoke({"messages": [{"role": "user", "content": "你好"}]}, config=config)
# print(f"Agent: {response['messages'][-1].content}")
# 
response = agent.invoke({"messages": [{"role": "user", "content": "介绍一下你自己"}]}, config=config)
# print(f"Agent: {response['messages'][-1].content}")

[日志] 消息数: 1
[日志] 响应类型: AIMessage
[日志] 消息数: 3
[日志] 响应类型: AIMessage


**计数中间件**

In [11]:
from langgraph.checkpoint.memory import InMemorySaver


class CallCounterMiddleware(AgentMiddleware):
    """
    计数中间件 - 统计模型调用次数

    在中间件内部维护计数器（简单版本）
    """

    def __init__(self):
        super().__init__()
        self.count = 0  # 简单计数器

    def after_model(self, state, runtime):
        """模型响应后，增加计数"""
        self.count += 1
        print(f"\n[计数器] 模型调用次数: {self.count}")
        return None  # 不修改 state

# 需要 checkpointer 来保存自定义状态
agent = create_agent(
    model=model,
    middleware=[CallCounterMiddleware()],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "counter_test"}}

print("第一次调用:")
agent.invoke({"messages": [{"role": "user", "content": "你好"}]}, config)

print("第二次调用:")
agent.invoke({"messages": [{"role": "user", "content": "今天天气"}]}, config)

第一次调用:

[计数器] 模型调用次数: 1
第二次调用:

[计数器] 模型调用次数: 2


{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='01dbbd8d-a2d0-4c9f-b5d4-6805acb4922b'),
  AIMessage(content='你好！很高兴见到你！😊 我是DeepSeek，由深度求索公司创造的AI助手。无论你有什么问题、需要什么帮助，或者只是想聊聊天，我都很乐意为你提供支持！\n\n我可以帮你处理各种任务，比如：\n- 回答问题和解释概念\n- 协助写作和翻译\n- 分析和处理文档\n- 编程和技术支持\n- 创意思考和头脑风暴\n\n有什么我可以为你做的吗？请随时告诉我你的需求！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 5, 'total_tokens': 104, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': 'e9ea1b40-2095-43a3-b974-3c3d7b9dfc8a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bd23a-fa3e-70a2-abbb-f8d5d5c7ded8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 99, 'tot

**消息修剪中间件**

In [6]:
class MessageTrimmerMiddleware(AgentMiddleware):
    """
    消息修剪中间件 - 限制消息数量

    before_model 修改消息列表
    注意：需要配合无 checkpointer 使用，否则历史会被恢复
    """

    def __init__(self, max_messages=5):
        super().__init__()
        self.max_messages = max_messages
        self.trimmed_count = 0  # 统计修剪次数

    def before_model(self, state, runtime):
        """模型调用前，修剪消息"""
        messages = state.get('messages', [])

        if len(messages) > self.max_messages:
            # 保留最近的 N 条消息
            trimmed_messages = messages[-self.max_messages:]
            self.trimmed_count += 1
            print(f"\n[修剪] 消息从 {len(messages)} 条减少到 {len(trimmed_messages)} 条 (第{self.trimmed_count}次修剪)")
            return {"messages": trimmed_messages}

        return None
        
middleware = MessageTrimmerMiddleware(max_messages=4)  # 最多保留 4 条
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
    middleware=[middleware]
    # 不使用 checkpointer
)

# 手动管理消息历史
messages = []
for i in range(6):
    print(f"\n--- 第 {i+1} 次对话 ---")

    # 新增用户消息
    new_msg = {"role": "user", "content": f"消息{i+1}：简短回复"}
    messages.append(new_msg)

    print(f"调用前消息数: {len(messages)}")

    # 调用 agent（middleware会修剪）
    response = agent.invoke({"messages": messages})

    # 获取完整对话（包含AI响应）
    messages = response['messages']

    print(f"调用后消息数: {len(messages)}")
    if len(messages) <= 4:
        print(f"消息列表: {[m.content[:15] for m in messages]}")


--- 第 1 次对话 ---
调用前消息数: 1
调用后消息数: 2
消息列表: ['消息1：简短回复', '好的，请讲。']

--- 第 2 次对话 ---
调用前消息数: 3
调用后消息数: 4
消息列表: ['消息1：简短回复', '好的，请讲。', '消息2：简短回复', '请告诉我您需要什么帮助。']

--- 第 3 次对话 ---
调用前消息数: 5

[修剪] 消息从 5 条减少到 4 条 (第1次修剪)
调用后消息数: 6

--- 第 4 次对话 ---
调用前消息数: 7

[修剪] 消息从 7 条减少到 4 条 (第2次修剪)
调用后消息数: 8

--- 第 5 次对话 ---
调用前消息数: 9

[修剪] 消息从 9 条减少到 4 条 (第3次修剪)
调用后消息数: 10

--- 第 6 次对话 ---
调用前消息数: 11

[修剪] 消息从 11 条减少到 4 条 (第4次修剪)
调用后消息数: 12


**输出验证中间件**

In [3]:
class OutputValidationMiddleware(AgentMiddleware):
    """
    输出验证中间件 - 检查响应长度

    after_model 验证输出
    """

    def __init__(self, max_length=100):
        super().__init__()
        self.max_length = max_length

    def after_model(self, state, runtime):
        """模型响应后，验证输出"""
        messages = state.get('messages', [])
        if not messages:
            return None

        last_message = messages[-1]
        content = getattr(last_message, 'content', '')

        if len(content) > self.max_length:
            print(f"\n[警告] 响应过长 ({len(content)} 字符)，已截断到 {self.max_length}")
            # 这里可以实现截断或重试逻辑
            # {}
        return None
        
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
    middleware=[OutputValidationMiddleware(max_length=50)]
)


print("\n用户: 请详细介绍 Python 编程语言的历史、特点和应用")
response = agent.invoke({
    "messages": [{"role": "user", "content": "请详细介绍 Python 编程语言的历史、特点和应用"}]
})
print(f"Agent: {response['messages'][-1].content[:50]}...")


用户: 请详细介绍 Python 编程语言的历史、特点和应用

[警告] 响应过长 (1964 字符)，已截断到 50
Agent: Python 是一种**高级、解释型、通用编程语言**，以其简洁的语法、强大的可读性和广泛的应用领域而闻名。以下是关于 Python 的详细介绍：

---

### **一、历史发展**
- **诞...


人工审核：**HumanInTheLoopMiddleware**

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv()

# 1. 初始化模型
model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1
)


# 2. 定义一个假的 send_email 工具
@tool
def send_email(to: str, subject: str, content: str) -> str:
    """
    发送邮件（示例工具，不会真的发送）
    """
    return (
        "【模拟发送成功】\n"
        f"收件人: {to}\n"
        f"主题: {subject}\n"
        f"内容: {content}"
    )



# 3. 创建带 Human-in-the-loop 的 Agent
agent = create_agent(
    model=model,
    tools=[send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True  # 在调用 send_email 前中断
            }
        )
    ]
)

# 必须指定 thread_id，否则无法 resume
config = {"configurable": {"thread_id": "hitl_demo"}}


# 4. 触发一个会调用 send_email 的请求

print("用户发起请求...")
response = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "帮我给 test@example.com 发一封邮件，主题是测试，内容是你好"
    }]},
    config=config
)

# print(response)

# 5. 此时流程已被中断（不会真正执行工具）
print("已触发 Human-in-the-loop，中断在工具调用前")
print("当前 Agent 输出：\n")
print(response["messages"][-1].content)


# 6. 模拟人工审批
choice = input("是否同意发送邮件？(y/n): ").strip().lower()

if choice != "y":
    print("人工拒绝了该操作")
else:
    print("人工同意，继续执行\n")

    last_msg = response["messages"][-1]
    tool_call = last_msg.tool_calls[0]
    tool_result = send_email.invoke(tool_call["args"])
    result = agent.invoke(
        {"messages": [
                {
                    "role": "tool",
                    "tool_call_id": tool_call["id"],
                    "content": tool_result
                }
            ]},
        config=config
    )

    print("Agent 最终回复：\n")
    print(result["messages"][-1].content)


用户发起请求...
已触发 Human-in-the-loop，中断在工具调用前
当前 Agent 输出：

我来帮您发送这封测试邮件。


是否同意发送邮件？(y/n):  y


人工同意，继续执行

Agent 最终回复：

邮件已成功发送！这是一封模拟邮件，实际不会真的发送出去。

收件人：test@example.com
主题：测试
内容：你好
